<div align="center">

# Symbolic Regression of CRDFT Reactivity Descriptors with PySR

### Genetic-programming baseline for the KAN benchmark: native operators, residual pruning, and seed dependence

*Supporting computational notebook, companion to* `KAN_from_CRDFT.ipynb`

</div>

---

**Abstract.** PySR is run on the same task and the same data as the Kolmogorov-Arnold network
(KAN) study: recover the closed forms of the eight global CRDFT descriptors from the
ionization potential, electron affinity and the features built from them. Its operator set contains
$\times$ and $\div$, so products and quotients are expressible natively, and its raw output is
pruned and rounded to the canonical form. The KAN numbers are read from the executed KAN
notebook rather than retyped, and the log-space least-squares anchor is recomputed here.

## 1. Scope and fairness of the comparison

A KAN sums learnable univariate functions and cannot represent a bare product or quotient, so
the KAN study maps the multiplicative descriptors to log-space; PySR evolves expression trees
over an operator set containing $\times$ and $\div$, so the multiplicative laws are directly
expressible. The comparison uses the same CSV and curation (with a guard on the molecule count,
3.2), the same features per descriptor, and the same success criterion: after post-processing the
expression must be symbolically identical to the reference. In the headline run PySR is offered
the unary `square` only where the reference contains a square, and 4.6 removes that prior.

## 2. Post-processing of PySR output

PySR returns the most accurate candidate on its Pareto front (`model_selection="accuracy"`), and
because the search optimizes constants numerically a correct *structure* typically arrives
carrying numerical residue. The cleaning rule, applied identically to every descriptor, sets
every numeric constant with $|c|<10^{-5}$ to zero and replaces the rest by the simplest rational
within $10^{-4}$ (`sympy.nsimplify`). Section 4.4 reports the largest pruned constant and the
full-data $R^2$ before and after cleaning.

## 3. Computational methods

Configuration, data loading and the exact anchor are identical in spirit to the KAN notebook.
`DRY=1` shrinks every budget for a quick end-to-end check.

**Cell 3.0. Install and companion modules.** PySR installs Julia and `SymbolicRegression.jl`
automatically on first import (a few minutes the first time on Colab). Upload `rsc_style.py`,
`figures_rsc.py`, `extract_kan_results.py` and the **executed** `KAN_from_CRDFT.ipynb` next to
this notebook.

In [ ]:
# 3.0 rsc_style.py and figures_rsc.py ship with this repository, not PyPI. The install is
# skipped when the environment already provides everything (clusters build it beforehand).
import os as _os, importlib.util as _ilu
_REQUIRED = ("pysr", "juliacall", "sympy", "numpy", "pandas", "matplotlib")
_missing = [m for m in _REQUIRED if _ilu.find_spec(m) is None]
if _os.environ.get("SKIP_PIP", "0") == "1" or not _missing:
    print("dependencies already satisfied; skipping pip install"
          + (" (SKIP_PIP=1)" if _os.environ.get("SKIP_PIP") == "1" else ""))
elif _os.path.exists("requirements-pysr.txt"):
    !pip -q install -r requirements-pysr.txt
else:
    !pip -q install pysr sympy scikit-learn pandas numpy matplotlib

import os, sys
sys.path.insert(0, os.getcwd())
_needed = [f for f in ("rsc_style.py", "figures_rsc.py", "extract_kan_results.py") if not os.path.exists(f)]
if _needed:
    try:
        from google.colab import files
        print("Missing:", ", ".join(_needed)); files.upload()
    except ImportError:
        raise FileNotFoundError(f"Put {', '.join(_needed)} next to this notebook and re-run.")
import importlib, rsc_style, figures_rsc
importlib.reload(rsc_style); importlib.reload(figures_rsc)
# Version guard. Compare as an INTEGER, not a string ("12" < "5" is true).
FIGURES_RSC_MIN = 12
_have = int(getattr(figures_rsc, "__version__", "0"))
if _have < FIGURES_RSC_MIN:
    raise ImportError(
        f"figures_rsc.py is v{_have}; this notebook needs v{FIGURES_RSC_MIN} or newer. "
        "Delete the old file from the file panel (or restart the runtime), upload the "
        "current one, and re-run this cell.")
for _fn in ("fig_method_comparison",):
    if not hasattr(figures_rsc, _fn):
        raise ImportError(f"figures_rsc.py v{_have} has no {_fn}(); upload the current file.")
print(f"companion modules loaded OK (figures_rsc v{figures_rsc.__version__})")

**Cell 3.1. Imports, configuration and environment.**

In [ ]:
import warnings, time, json, hashlib, platform
import numpy as np, pandas as pd, sympy as sp
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore", category=FutureWarning, module="pysr")

DRY = os.environ.get("DRY", "0") == "1"
EXPECTED_N = int(os.environ.get("EXPECTED_N", "131375"))   # curated set of the paper (Section 3.1)
if DRY:
    NITER, SUBSAMPLE, SEEDS = 5, 500, [0, 1]
else:
    NITER     = 60      # PySR iterations per fit
    SUBSAMPLE = 2000    # molecules used by the search; R2 is always evaluated on all of them
    SEEDS     = [0, 1, 2]   # seed 0 is the headline run; 1 and 2 form the seed study
MAXSIZE  = 18
PRUNE_TOL, SNAP_TOL, REL_TOL = 1e-5, 1e-4, 1e-6
print(("DRY" if DRY else "FULL") + f" configuration | niterations={NITER} | subsample={SUBSAMPLE} "
      f"| maxsize={MAXSIZE} | seeds={SEEDS}")

import importlib.metadata as _md
def _ver(p):
    try: return _md.version(p)
    except Exception: return "not installed"
print(f"\nEnvironment\n  python   {platform.python_version()}  ({platform.platform()})")
for _p in ["pysr", "juliacall", "sympy", "numpy", "pandas", "matplotlib"]:
    print(f"  {_p:<10} {_ver(_p)}")
from pysr import PySRRegressor
try:
    from juliacall import Main as _jl
    print(f"  julia      {_jl.seval('string(VERSION)')}")
    print(f"  SymbolicRegression.jl {_jl.seval('using SymbolicRegression; string(pkgversion(SymbolicRegression))')}")
except Exception as e:
    print(f"  julia      (version query failed: {e!r})")

**Cell 3.2. Load and curate the dataset.** Identical to the KAN notebook. The SHA-256 of the
CSV is printed so the file can be matched to the archived one, and the notebook **stops** if the
curated molecule count differs from `EXPECTED_N`.

In [ ]:
CSV_PATH = "reactivity_dataset.csv"
USED_SYNTHETIC = not os.path.exists(CSV_PATH)
if not USED_SYNTHETIC:
    DATA_SHA256 = hashlib.sha256(open(CSV_PATH, "rb").read()).hexdigest()
    df = pd.read_csv(CSV_PATH)
else:
    DATA_SHA256 = None
    warnings.warn("reactivity_dataset.csv not found -- using SYNTHETIC data. Results below do "
                  "NOT reproduce the manuscript.", RuntimeWarning)
    print("=" * 74); print("  WARNING: SYNTHETIC DATA IN USE -- results are NOT reproducible"); print("=" * 74)
    rng = np.random.default_rng(7); N = 131385
    IPg = np.clip(rng.normal(8.56, 1.14, N), 3.69, 15.89); EAg = np.clip(rng.normal(-0.46, 1.28, N), -4.45, 4.57)
    etag = IPg - EAg; m = etag > 0.05; IPg, EAg, etag = IPg[m], EAg[m], etag[m]
    mug = -(IPg + EAg) / 2; chig = (IPg + EAg) / 2
    df = pd.DataFrame({"IP": IPg, "EA": EAg, "mu": mug, "chi": chig, "eta": etag, "S": 1 / etag,
                       "Nmax": chig / etag, "omega": mug**2 / (2 * etag),
                       "omega_plus": (IPg + 3 * EAg)**2 / (16 * etag),
                       "omega_minus": (3 * IPg + EAg)**2 / (16 * etag)})

df = df[df["eta"] > 0].replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
IP, EA, mu, eta = df.IP.values, df.EA.values, df.mu.values, df.eta.values
chi = -mu; Ap, Am = np.abs(IP + 3 * EA), np.abs(3 * IP + EA)
N_MOL = len(df)
print(f"Molecules: {N_MOL}   (source: {'SYNTHETIC' if USED_SYNTHETIC else CSV_PATH})")
if DATA_SHA256: print(f"CSV sha256: {DATA_SHA256}")
if not USED_SYNTHETIC and N_MOL != EXPECTED_N:
    raise RuntimeError(f"{CSV_PATH} gives {N_MOL} molecules after curation, but the paper's data set "
                       f"has {EXPECTED_N}. This is a different file; fix it (or set EXPECTED_N if intended).")
df[["IP","EA","mu","chi","eta","S","Nmax","omega","omega_plus","omega_minus"]].describe().T[["mean","std","min","max"]].round(4)

**Cell 3.3. Sanity check: the reference identities.**

In [ ]:
checks = {"mu": -(IP+EA)/2, "chi": (IP+EA)/2, "eta": IP-EA, "S": 1/eta, "Nmax": chi/eta,
          "omega": mu**2/(2*eta), "omega_plus": (IP+3*EA)**2/(16*eta), "omega_minus": (3*IP+EA)**2/(16*eta)}
for k, v in checks.items(): print(f"{k:12s} max|res| = {np.abs(v - df[k].values).max():.2e}")

**Cell 3.4. Tasks, metrics and the cleaning rule.**

In [ ]:
IP_s, EA_s, chi_s, eta_s, Ap_s, Am_s = sp.symbols("IP EA chi eta A_plus A_minus")
# (name, target column, input symbols, input arrays, unary operators for the headline run)
TASKS = [("mu", "mu", [IP_s, EA_s], [IP, EA], []),
         ("chi", "chi", [IP_s, EA_s], [IP, EA], []),
         ("eta", "eta", [IP_s, EA_s], [IP, EA], []),
         ("S", "S", [eta_s], [eta], []),
         ("Nmax", "Nmax", [chi_s, eta_s], [chi, eta], []),
         ("omega", "omega", [chi_s, eta_s], [chi, eta], ["square"]),
         ("omega_plus", "omega_plus", [Ap_s, eta_s], [Ap, eta], ["square"]),
         ("omega_minus", "omega_minus", [Am_s, eta_s], [Am, eta], ["square"])]
REF = {"mu": -(IP_s+EA_s)/2, "chi": (IP_s+EA_s)/2, "eta": IP_s-EA_s, "S": 1/eta_s,
       "Nmax": chi_s/eta_s, "omega": chi_s**2/(2*eta_s),
       "omega_plus": Ap_s**2/(16*eta_s), "omega_minus": Am_s**2/(16*eta_s)}
NAMES = [t[0] for t in TASKS]

def ops(expr):
    return int(sp.count_ops(expr))

def r2_full(expr, syms, arrays, col):
    f = sp.lambdify(syms, expr, "numpy")
    pred = np.broadcast_to(f(*arrays), df[col].values.shape).astype(float)
    y = df[col].values
    return float(1 - np.var(y - pred) / np.var(y))

def _rms(expr, syms, arrays):
    """Root-mean-square value of an expression over the full molecule set."""
    f = sp.lambdify(syms, expr, "numpy")
    v = np.broadcast_to(np.asarray(f(*arrays), dtype=float), (len(arrays[0]),))
    return float(np.sqrt(np.mean(v**2)))

def _drop_negligible(expr, syms, arrays, rel_tol):
    """Remove any term of a sum whose RMS contribution is below rel_tol times the RMS
    of that sum. The test is relative, so it does not depend on the units of the
    descriptor, and it applies to additive terms rather than to numeric literals."""
    worst = 0.0
    def rec(e):
        nonlocal worst
        if e.args:
            e = e.func(*[rec(a) for a in e.args])
        if e.is_Add:
            scale = _rms(e, syms, arrays)
            if scale > 0.0:
                keep, drop = [], []
                for t in e.args:
                    (keep if _rms(t, syms, arrays) >= rel_tol*scale else drop).append(t)
                if keep and drop:
                    for t in drop:
                        worst = max(worst, _rms(t, syms, arrays)/scale)
                    e = sp.Add(*keep)
        return e
    return rec(expr), worst

def clean_expr(expr, syms, arrays, rel_tol=REL_TOL, ctol=SNAP_TOL):
    """Drop negligible additive terms, snap the surviving constants to the simplest
    rational within ctol, then drop again: rationalizing two nearly cancelling
    constants can introduce a residue the first pass never sees.
    Returns (cleaned expression, largest relative contribution dropped)."""
    e, w1 = _drop_negligible(expr, syms, arrays, rel_tol)
    e = e.xreplace({a: sp.nsimplify(a, rational=True, tolerance=ctol)
                    for a in e.atoms(sp.Float)})
    e, w2 = _drop_negligible(sp.together(e), syms, arrays, rel_tol)
    return sp.simplify(e), max(w1, w2)

def clean_expr_abs(expr, tol=PRUNE_TOL, ctol=SNAP_TOL):
    """The absolute-threshold rule, kept only so its failure rate can be reported."""
    reps, pruned = {}, 0.0
    for a in expr.atoms(sp.Float):
        if abs(float(a)) < tol:
            reps[a] = sp.Integer(0); pruned = max(pruned, abs(float(a)))
        else:
            reps[a] = sp.nsimplify(a, rational=True, tolerance=ctol)
    return sp.simplify(expr.xreplace(reps)), pruned

def matches(expr, name):
    return sp.simplify(expr - REF[name]) == 0

print("canonical operation counts:", {k: ops(v) for k, v in REF.items()})

## 4. Results

### 4.1 The exact least-squares anchor (recomputed here)

**Cell 4.1.** Linear least squares on $(IP,EA)$ for the affine descriptors and log-space least
squares on the features for the others, rounded exactly as in the KAN notebook. This is the upper
bound every method is measured against, and its cost.

In [ ]:
def _snap_half(v): return sp.Rational(round(2*float(v)), 2)
PHYS = [sp.Integer(1), sp.Rational(1,2), sp.Rational(1,4), sp.Rational(1,8), sp.Rational(1,16),
        sp.Integer(2), sp.Integer(3), sp.Integer(16)]
def _snap_const(v):
    c = min(PHYS + [-x for x in PHYS], key=lambda z: abs(float(z) - v))
    return c if abs(float(c) - v) < 0.03*max(abs(v), 1e-9) else sp.nsimplify(v, tolerance=0.01)

anchor = {}
for name, col, syms, arrays, _ in TASKS:
    y = df[col].values
    t0 = time.perf_counter()
    if name in ("mu", "chi", "eta"):
        A = np.column_stack(arrays + [np.ones(N_MOL)])
        c, *_ = np.linalg.lstsq(A, y, rcond=None)
        const = _snap_half(c[-1]) if abs(c[-1]) > 0.05 else sp.Integer(0)
        expr = sp.simplify(sum(_snap_half(ci)*s for ci, s in zip(c[:-1], syms)) + const)
    else:
        g = y > 1e-12
        for a in arrays: g &= np.abs(a) > 1e-9
        A = np.column_stack([np.log(np.abs(a[g])) for a in arrays] + [np.ones(int(g.sum()))])
        c, *_ = np.linalg.lstsq(A, np.log(y[g]), rcond=None)
        expr = _snap_const(float(np.exp(c[-1])))
        for ci, s in zip(c[:-1], syms): expr *= s**_snap_half(ci)
        expr = sp.simplify(expr)
    wall = time.perf_counter() - t0
    anchor[name] = dict(expr=expr, match=matches(expr, name), r2=r2_full(expr, syms, arrays, col), wall=wall)
    print(f"{name:12s} {str(expr):24s} match={'yes' if anchor[name]['match'] else 'NO':3s} "
          f"R2={anchor[name]['r2']:.6f}  time={1e3*wall:.1f} ms")

### 4.2 KAN results (read from the executed KAN notebook)

**Cell 4.2.** `results/kan_results.json` is written by `extract_kan_results.py`, which parses the
printed outputs of the executed `KAN_from_CRDFT.ipynb`. The cell stops if that file was produced
on a data set of a different size.

In [ ]:
import glob, subprocess
KAN_JSON = "results/kan_results.json"
if not os.path.exists(KAN_JSON):
    # Colab renames re-uploaded files, so accept any variant.
    cands = sorted(glob.glob("KAN_from_CRDFT*.ipynb"), key=os.path.getmtime, reverse=True)
    # Prefer a candidate that has outputs: the run-me notebook ships with them cleared.
    def _has_outputs(f):
        try:
            nb_ = json.load(open(f))
        except Exception:
            return False
        return any(c.get("outputs") for c in nb_.get("cells", []))
    cands = [c for c in cands if _has_outputs(c)] or cands
    if not cands:
        try:
            from google.colab import files
            print("Upload the EXECUTED KAN notebook (run on the 131,375-molecule CSV):")
            cands = [f for f in files.upload() if f.endswith(".ipynb")]
        except ImportError:
            pass
    if not cands:
        raise FileNotFoundError("No executed KAN notebook found. .ipynb files here: "
                                f"{sorted(glob.glob('*.ipynb'))}. Upload KAN_from_CRDFT.ipynb "
                                "(or results/kan_results.json) and re-run this cell.")
    print(f"extracting KAN results from: {cands[0]}")
    r = subprocess.run([sys.executable, "extract_kan_results.py", cands[0]], capture_output=True, text=True)
    print(r.stdout)
    if r.returncode:
        raise RuntimeError("extract_kan_results.py failed -- is that notebook executed (with outputs)?\n" + r.stderr[-800:])
kan = json.load(open(KAN_JSON))
if kan["n_molecules"] != N_MOL:
    msg = (f"KAN results come from a run on {kan['n_molecules']} molecules, this notebook has {N_MOL}. "
           "They are not comparable.")
    if USED_SYNTHETIC: warnings.warn(msg + " (tolerated only because synthetic data is in use)")
    else: raise RuntimeError(msg)
print(f"KAN results from {kan['source_notebook']} (N={kan['n_molecules']}); "
      f"network alone recovers {kan['network_only_recovered']}/8\n")
KAN = {}
for name in NAMES:
    k = kan["descriptors"][name]
    expr = sp.sympify(k["network_closed_form"], locals={"IP": IP_s, "EA": EA_s, "chi": chi_s,
                      "eta": eta_s, "A_plus": Ap_s, "A_minus": Am_s})
    KAN[name] = dict(expr=expr, match=matches(expr, name), wall=k["wall_s"], raw=k["network_raw"])
    print(f"{name:12s} {str(expr):24s} match={'yes' if KAN[name]['match'] else 'NO':3s} "
          f"time={k['wall_s']:.1f} s   network: {k['network_raw']}")

### 4.3 PySR, headline run

**Cell 4.3.** Binary operators $\{+,-,\times,\div\}$, the unary `square` where the reference
contains a square, squared-error loss, `maxsize=18`, the most accurate Pareto candidate, and a
deterministic serial search with a fixed seed. The search sees `SUBSAMPLE` molecules; $R^2$ is
evaluated on all of them.

In [ ]:
def run_pysr(X, y, unary, seed, niter=NITER, subsample=SUBSAMPLE):
    idx = np.random.RandomState(seed).choice(len(X), min(subsample, len(X)), replace=False)
    model = PySRRegressor(
        niterations=niter, maxsize=MAXSIZE,
        binary_operators=["+", "-", "*", "/"], unary_operators=list(unary),
        elementwise_loss="loss(x, y) = (x - y)^2", model_selection="accuracy",
        deterministic=True, parallelism="serial", random_state=seed,
        progress=False, verbosity=0, temp_equation_file=True)
    t0 = time.time(); model.fit(X[idx], y[idx]); wall = time.time() - t0
    return model, wall

def pysr_fit(name, col, syms, arrays, unary, seed):
    X = np.column_stack(arrays); y = df[col].values
    model, wall = run_pysr(X, y, unary, seed)
    raw = sp.sympify(model.sympy()).subs({sp.Symbol(f"x{i}"): s for i, s in enumerate(syms)})
    clean, pruned = clean_expr(raw, syms, arrays)
    clean_a, _ = clean_expr_abs(raw)
    return dict(raw=raw, ops_raw=ops(raw), r2_raw=r2_full(raw, syms, arrays, col),
                clean=clean, ops_clean=ops(clean), r2_clean=r2_full(clean, syms, arrays, col),
                pruned=pruned, match=matches(clean, name), wall=wall,
                clean_abs=clean_a, match_abs=matches(clean_a, name))

PY = {}
for name, col, syms, arrays, unary in TASKS:
    PY[name] = pysr_fit(name, col, syms, arrays, unary, seed=SEEDS[0])
    p = PY[name]
    print(f"{name:12s} raw: {str(p['raw']):40s} ops={p['ops_raw']} R2={p['r2_raw']:.6f} time={p['wall']:.0f} s")

### 4.4 Pruning the residuals

**Cell 4.4.** The cleaning rule of Section 2, with the largest constant it removed and $R^2$
before and after. If pruning removed anything that mattered, $R^2$ would drop.

In [ ]:
print(f"{'descriptor':12s} {'cleaned':24s} {'match':5s} {'largest pruned':>15s} {'R2 raw':>10s} {'R2 clean':>10s} {'ops raw->clean':>15s}")
for name in NAMES:
    p = PY[name]
    print(f"{name:12s} {str(p['clean']):24s} {'yes' if p['match'] else 'NO':5s} "
          f"{p['pruned']:15.2e} {p['r2_raw']:10.6f} {p['r2_clean']:10.6f} {p['ops_raw']:>7d} -> {p['ops_clean']}")
n_py = sum(PY[n]["match"] for n in NAMES)
n_abs = sum(PY[n]["match_abs"] for n in NAMES)
print(f"absolute-threshold rule, for reference: {n_abs}/8")
print(f"\nPySR (seed {SEEDS[0]}), after pruning: {n_py}/8 closed forms recovered.")

### 4.5 Seed dependence

**Cell 4.5.** PySR's search is stochastic; with `deterministic=True` it is reproducible for a
given seed, but different seeds explore different trees. Every fit is repeated with the
remaining seeds, recording whether the pruned expression still matches.

In [ ]:
SEED_RUNS = {SEEDS[0]: PY}
for s in SEEDS[1:]:
    SEED_RUNS[s] = {name: pysr_fit(name, col, syms, arrays, unary, seed=s)
                    for name, col, syms, arrays, unary in TASKS}
print(f"{'descriptor':12s} " + " ".join(f"{'seed '+str(s):>18s}" for s in SEEDS))
for name in NAMES:
    cells = []
    for s in SEEDS:
        p = SEED_RUNS[s][name]
        cells.append(f"{'yes' if p['match'] else 'NO '} ({p['wall']:4.0f} s)")
    print(f"{name:12s} " + " ".join(f"{c:>18s}" for c in cells))
per_seed = {s: sum(SEED_RUNS[s][n]["match"] for n in NAMES) for s in SEEDS}
print("\nrecovered per seed:", per_seed)
for s in SEEDS:
    for n in NAMES:
        if not SEED_RUNS[s][n]["match"]:
            print(f"  seed {s}, {n}: raw {SEED_RUNS[s][n]['raw']}  ->  {SEED_RUNS[s][n]['clean']}")

### 4.6 Operator-set ablation: `square` for every descriptor

**Cell 4.6.** In 4.3 `square` was offered only where the reference contains a square. Here it is
offered for all eight, removing that prior.

In [ ]:
ABL = {name: pysr_fit(name, col, syms, arrays, ["square"], seed=SEEDS[0])
       for name, col, syms, arrays, _ in TASKS}
for name in NAMES:
    p = ABL[name]
    print(f"{name:12s} {str(p['clean']):24s} {'yes' if p['match'] else 'NO':3s}  raw: {p['raw']}")
print(f"\nWith `square` offered for all descriptors: {sum(ABL[n]['match'] for n in NAMES)}/8 recovered.")

### 4.7 Head-to-head comparison

**Cell 4.7 (Table and Figure C1).** Anchor, KAN (network alone) and PySR (seed 0, pruned) side
by side. "ops" is the SymPy operation count of the canonical form, "ops raw" that of PySR's
unpruned output. The table is also written to `results/pysr_results.json`.

In [ ]:
rows = []
for name, col, syms, arrays, _ in TASKS:
    rows.append(dict(name=name, ops_canonical=ops(REF[name]),
                     anchor=anchor[name]["match"], KAN=KAN[name]["match"], PySR=PY[name]["match"],
                     ops_raw_PySR=PY[name]["ops_raw"], r2_PySR=PY[name]["r2_clean"],
                     wall_anchor=anchor[name]["wall"], wall_KAN=KAN[name]["wall"], wall_PySR=PY[name]["wall"],
                     pysr_seeds_recovered=sum(SEED_RUNS[s][name]["match"] for s in SEEDS),
                     pysr_square_all=ABL[name]["match"]))
tab = pd.DataFrame(rows).set_index("name")
show = tab[["anchor","KAN","PySR","ops_canonical","ops_raw_PySR","wall_anchor","wall_KAN","wall_PySR",
            "pysr_seeds_recovered","pysr_square_all"]].copy()
show["wall_anchor"] = (1e3*show["wall_anchor"]).round(2).astype(str) + " ms"
show["wall_KAN"] = show["wall_KAN"].round(1).astype(str) + " s"
show["wall_PySR"] = show["wall_PySR"].round(1).astype(str) + " s"
show["pysr_seeds_recovered"] = show["pysr_seeds_recovered"].astype(str) + f"/{len(SEEDS)}"
display(show)

os.makedirs("results", exist_ok=True)
json.dump(dict(n_molecules=N_MOL, data_sha256=DATA_SHA256, synthetic=USED_SYNTHETIC,
               config=dict(niterations=NITER, subsample=SUBSAMPLE, maxsize=MAXSIZE, seeds=SEEDS,
                           prune_tol=PRUNE_TOL, snap_tol=SNAP_TOL, rel_tol=REL_TOL),
               descriptors={n: dict(raw=str(PY[n]["raw"]), clean=str(PY[n]["clean"]), match=bool(PY[n]["match"]),
                                    ops_raw=PY[n]["ops_raw"], ops_canonical=ops(REF[n]), pruned=PY[n]["pruned"], match_abs=bool(PY[n]["match_abs"]),
                                    clean_abs=str(PY[n]["clean_abs"]),
                                    r2_raw=PY[n]["r2_raw"], r2_clean=PY[n]["r2_clean"], wall_s=PY[n]["wall"],
                                    seeds={str(s): bool(SEED_RUNS[s][n]["match"]) for s in SEEDS},
                                    square_all=bool(ABL[n]["match"]),
                                    anchor_wall_s=anchor[n]["wall"]) for n in NAMES}),
          open("results/pysr_results.json", "w"), indent=2)
print("wrote results/pysr_results.json")

import rsc_style as rsc, figures_rsc as F
rsc.use()
F.fig_method_comparison(rows, methods=("anchor", "KAN", "PySR"), stem="figC4_pysr_comparison");